In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir("/content/drive/MyDrive/part_c_diffusion")
sys.path.append(os.getcwd())

!pip install -q diffusers

In [ ]:
!apt-get -qq install unrar
!mkdir -p /content/uhat_data
!unrar x -inul /content/drive/MyDrive/uhat.rar /content/uhat_data/
!ls /content/uhat_data
DATA_DIR = "/content/uhat_data/uhat"   # if everything is inside one extra folder, add it here, e.g. /content/uhat_data/uhat

In [ ]:
!apt-get -qq install unrar
!mkdir -p /content/uhat_data
!unrar x -inul /content/drive/MyDrive/uhat.rar /content/uhat_data/

import numpy as np
DATA_DIR = "/content/uhat_data/uhat"
d = np.load(f"{DATA_DIR}/uhat_dataset.npz", allow_pickle=True)
for k in d.files:
    print(k, d[k].shape, d[k].dtype)

In [ ]:
# 2. Load data, build model, run the checks
import torch
from data import get_data
from model import build_unet
from diffusion import DEVICE, alpha_bar

loader, images, class_names = get_data()
unet = build_unet(DEVICE)

print("alpha_bar[0], alpha_bar[-1]:", alpha_bar[0].item(), alpha_bar[-1].item())  # ~1, ~4e-5
x0, y = next(iter(loader))
print("data range:", x0.min().item(), x0.max().item(), "| label dtype:", y.dtype)

x = torch.randn(8, 1, 28, 28, device=DEVICE)
t = torch.randint(0, 1000, (8,), device=DEVICE).long()
yy = torch.randint(0, 40, (8,), device=DEVICE).long()
print("U-Net output:", unet(x, t, class_labels=yy).sample.shape)                  # (8,1,28,28)
print("Parameters:", sum(p.numel() for p in unet.parameters()))

In [ ]:
# 3. Train (try epochs=2 first, then 40)
from train import train
train(unet, loader, epochs=40, ckpt_path="/content/drive/MyDrive/part_c_ddpm.pt")

In [ ]:
# 4. Figures
from evaluate import forward_figure, grid_figure, denoising_figure
forward_figure(images)
grid_figure(unet, class_names)
denoising_figure(unet, class_id=5)

In [ ]:
# 5. Blind human eval
from evaluate import make_eval_sheet, score_eval
make_eval_sheet(unet, class_names)
# fill results/human_eval.csv (id, member1, member2), then:
# score_eval()